# 1. Header, Citations, Purpose & Topic Bridges

**Topic:** Vision Detection: Anchor-Free FCOS & Set-Based DETR
**Category:** 07_domain_tasks

### Citations & Core References
- Tian, Z., Shen, C., Chen, H., & He, T. (2019). *FCOS: Fully Convolutional One-Stage Object Detection*. ICCV.
- Rezatofighi, H., et al. (2019). *Generalized Intersection over Union: A Metric and A Loss for Bounding Box Regression*. CVPR.
- Carion, N., Massa, F., Synnaeve, G., Usunier, N., Kirillov, A., & Zagoruyko, S. (2020). *End-to-End Object Detection with Transformers (DETR)*. ECCV.
- Kuhn, H. W. (1955). *The Hungarian Method for the Assignment Problem*. Naval Research Logistics Quarterly.
- Zhu, X., et al. (2020). *Deformable DETR: Deformable Transformers for End-to-End Object Detection*. ICLR.

## 2. Mathematical Foundations & Governing Equations

### FCOS Centerness Metric
For location $(x, y)$ inside bounding box with distance offsets $[l, t, r, b]$:
$$\text{centerness} = \sqrt{\frac{\min(l, r)}{\max(l, r)} \times \frac{\min(t, b)}{\max(t, b)}}.$$

### Generalized IoU (GIoU) Metric
$$\text{GIoU} = \frac{|A \cap B|}{|A \cup B|} - \frac{|C \setminus (A \cup B)|}{|C|} = \text{IoU} - \frac{|C| - |A \cup B|}{|C|},$$
where $C$ is the smallest convex bounding box enclosing both $A$ and $B$.

### DETR Hungarian Matching Cost
Finds optimal 1-to-1 matching $\sigma \in \mathfrak{S}_N$ minimizing pairwise cost:
$$\mathcal{L}_{\text{match}}(y_i, \hat{y}_j) = -\hat{p}_j(c_i) + \lambda_{\text{bbox}} \|b_i - \hat{b}_j\|_1 + \lambda_{\text{giou}} \mathcal{L}_{\text{giou}}(b_i, \hat{b}_j).$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import torch
import numpy as np

torch.manual_seed(42)
B, N_loc, N_q, N_gt, Num_Classes = 2, 4, 6, 3, 5

# 1. FCOS distance targets [l, t, r, b]
ltrb_targets = torch.abs(torch.randn(B, N_loc, 4)) * 20.0 + 1.0

# 2. Bounding boxes in [x1, y1, x2, y2]
boxes_base = torch.rand(N_loc, 2) * 50.0
boxes1 = torch.cat([boxes_base, boxes_base + torch.rand(N_loc, 2) * 40.0 + 5.0], dim=-1)
boxes2 = torch.cat([boxes_base + torch.randn(N_loc, 2) * 5.0, boxes_base + torch.rand(N_loc, 2) * 50.0 + 10.0], dim=-1)

# 3. DETR queries and ground truth
logits = torch.randn(N_q, Num_Classes)
pred_prob = torch.softmax(logits, dim=-1)
pred_boxes = torch.rand(N_q, 4) * 0.8 + 0.1 # [cx, cy, w, h]
gt_classes = torch.tensor([1, 3, 2], dtype=torch.long)
gt_boxes = torch.tensor([
    [0.2, 0.3, 0.2, 0.4],
    [0.5, 0.5, 0.3, 0.3],
    [0.7, 0.2, 0.2, 0.2],
])

print(f"FCOS locations: {N_loc}, DETR queries: {N_q}, Ground-truth objects: {N_gt}")

In [ ]:
# 4. First-Principles Implementation & Native Parity
# 1. FCOS Centerness
def fcos_centerness(ltrb):
    l, t, r, b = ltrb[..., 0], ltrb[..., 1], ltrb[..., 2], ltrb[..., 3]
    lr_term = torch.minimum(l, r) / torch.clamp(torch.maximum(l, r), min=1e-6)
    tb_term = torch.minimum(t, b) / torch.clamp(torch.maximum(t, b), min=1e-6)
    return torch.sqrt(torch.clamp(lr_term * tb_term, min=0.0, max=1.0))

centerness_val = fcos_centerness(ltrb_targets)

# 2. Generalized IoU
def generalized_iou(box1, box2):
    x1_i = torch.maximum(box1[:, 0], box2[:, 0])
    y1_i = torch.maximum(box1[:, 1], box2[:, 1])
    x2_i = torch.minimum(box1[:, 2], box2[:, 2])
    y2_i = torch.minimum(box1[:, 3], box2[:, 3])
    area_inter = torch.clamp(x2_i - x1_i, min=0.0) * torch.clamp(y2_i - y1_i, min=0.0)
    area1 = torch.clamp(box1[:, 2] - box1[:, 0], min=0.0) * torch.clamp(box1[:, 3] - box1[:, 1], min=0.0)
    area2 = torch.clamp(box2[:, 2] - box2[:, 0], min=0.0) * torch.clamp(box2[:, 3] - box2[:, 1], min=0.0)
    area_union = area1 + area2 - area_inter
    iou = area_inter / torch.clamp(area_union, min=1e-6)
    
    x1_c = torch.minimum(box1[:, 0], box2[:, 0])
    y1_c = torch.minimum(box1[:, 1], box2[:, 1])
    x2_c = torch.maximum(box1[:, 2], box2[:, 2])
    y2_c = torch.maximum(box1[:, 3], box2[:, 3])
    area_c = torch.clamp(x2_c - x1_c, min=0.0) * torch.clamp(y2_c - y1_c, min=0.0)
    return iou - (area_c - area_union) / torch.clamp(area_c, min=1e-6)

giou_val = generalized_iou(boxes1, boxes2)

# 3. DETR Bipartite Matching Cost Matrix
def box_cxcywh_to_xyxy(b):
    cx, cy, w, h = b.unbind(-1)
    return torch.stack([cx - 0.5 * w, cy - 0.5 * h, cx + 0.5 * w, cy + 0.5 * h], dim=-1)

cost_cls = -pred_prob[:, gt_classes]
cost_l1 = torch.cdist(pred_boxes, gt_boxes, p=1.0)
b1_xyxy = box_cxcywh_to_xyxy(pred_boxes)
b2_xyxy = box_cxcywh_to_xyxy(gt_boxes)
cost_giou_mat = torch.zeros(N_q, N_gt)
for i in range(N_q):
    cost_giou_mat[i] = -generalized_iou(b1_xyxy[i:i+1].expand(N_gt, 4), b2_xyxy)
total_cost = 1.0 * cost_cls + 5.0 * cost_l1 + 2.0 * cost_giou_mat

print(f"Centerness mean: {torch.mean(centerness_val):.4f}, GIoU min/max: {torch.min(giou_val):.4f}/{torch.max(giou_val):.4f}")
print(f"Hungarian Cost Matrix shape: {total_cost.shape}, min cost: {torch.min(total_cost):.4f}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification
# Invariant 1: Centerness strictly in [0, 1]
assert torch.all((centerness_val >= 0.0) & (centerness_val <= 1.0)), "Centerness out of bounds"

# Invariant 2: GIoU bounded in [-1, 1]
assert torch.all((giou_val >= -1.0) & (giou_val <= 1.0)), "GIoU metric out of theoretical bounds [-1, 1]"

# Invariant 3: Perfect self-overlap yields GIoU = 1.0
giou_self = generalized_iou(boxes1, boxes1)
assert torch.max(torch.abs(giou_self - 1.0)).item() < 1e-6, "Self-overlap GIoU did not equal 1.0"

print("All vision detection invariants (centerness, GIoU bounds, self-overlap) verified successfully.")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

iters = 200
t0 = time.perf_counter()
for _ in range(iters):
    _ = fcos_centerness(ltrb_targets)
dt = (time.perf_counter() - t0) / iters * 1000.0
print(f"FCOS centerness evaluation latency: {dt:.3f} ms / step")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation
# Pathological: Standard IoU on non-overlapping boxes produces zero gradient everywhere
disjoint_b1 = torch.tensor([[0.0, 0.0, 10.0, 10.0]], requires_grad=True)
disjoint_b2 = torch.tensor([[50.0, 50.0, 60.0, 60.0]])
iou_zero = (torch.clamp(torch.minimum(disjoint_b1[:, 2], disjoint_b2[:, 2]) - torch.maximum(disjoint_b1[:, 0], disjoint_b2[:, 0]), min=0.0)).sum()
iou_zero.backward()
print(f"Pathological standard IoU gradient on disjoint boxes: {disjoint_b1.grad} (Zero gradient, learning stalls!)")

# Corrected: GIoU penalizes empty enclosing volume, providing non-vanishing gradients toward convergence
disjoint_b1_corr = torch.tensor([[0.0, 0.0, 10.0, 10.0]], requires_grad=True)
giou_loss = (1.0 - generalized_iou(disjoint_b1_corr, disjoint_b2)).sum()
giou_loss.backward()
print(f"Corrected GIoU non-vanishing gradient: {disjoint_b1_corr.grad.norm().item():.4f}")

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Anchor-Free Efficiency**: FCOS replaces thousands of rigid anchor boxes with direct per-pixel distance regression, using centerness to filter low-quality predictions.
2. **Global Set Prediction**: DETR formulates detection as bipartite matching via the Hungarian algorithm, removing heuristic Non-Maximum Suppression (NMS).
3. **Non-Vanishing Geometric Gradients**: Generalized IoU (GIoU) provides informative loss gradients even when predicted bounding boxes are completely disjoint from ground truth.